# Telemachus Quickstart

Ce notebook montre comment utiliser le SDK Python **Telemachus** pour :
- charger des données JSON/JSONL,
- valider contre le schéma Core,
- calculer le **Telemachus Completeness Score (TCS)**,
- exporter en **Parquet** puis recharger.

Il crée d'abord un petit fichier d'exemple local pour fonctionner **sans dépendance externe**.

In [1]:
# 0) Préparer un exemple local autonome
import os, json, textwrap
os.makedirs("examples", exist_ok=True)

example_path = "examples/geotab.json"

# Un enregistrement minimal valide (GNSS + speed + event)
record = {
    "timestamp": "2025-01-01T12:00:00Z",
    "vehicle_id": "GEOTAB-01",
    "position": {"lat": 48.85, "lon": 2.35, "altitude_m": 35.0, "heading_deg": 180.0},
    "motion": {"speed_kph": 45.2},
    "events": [{"type": "harsh_brake", "severity": "high", "start": "2025-01-01T12:00:10Z", "end": "2025-01-01T12:00:12Z"}],
    "source": {"provider": "geotab", "device_id": "DEVICE-9876", "ingest_timestamp": "2025-01-01T12:00:05Z"}
}

with open(example_path, "w") as f:
    json.dump(record, f)

example_path

'examples/geotab.json'

In [2]:
# 1) Charger en DataFrame
from telemachus.io import load_jsonl
df = load_jsonl(example_path)
df.head()

,timestamp,vehicle_id,events,position.lat,position.lon,position.altitude_m,position.heading_deg,motion.speed_kph,source.provider,source.device_id,source.ingest_timestamp
0,2025-01-01T12:00:00Z,GEOTAB-01,"[{'type': 'harsh_brake', 'severity': 'high', '...",48.85,2.35,35.0,180.0,45.2,geotab,DEVICE-9876,2025-01-01T12:00:05Z


In [3]:
# 2) Valider contre le schéma Telemachus Core
from telemachus.validate import validate

# Par défaut, la fonction va chercher le schéma sur RAW GitHub (configurée dans le SDK)
report = validate(example_path)
report  # {"ok": True/False, "errors": [...]} si erreurs

{'ok': True, 'errors': []}

In [4]:
# 3) Calculer le Telema(ch)us Completeness Score (TCS)
from telemachus.validate import score_completeness

tcs = score_completeness(df)
print(f"TCS global: {tcs['score_pct']:.1f}%")
list(tcs["coverage"].items())[:8]  # aperçu des premiers champs (couverture par champ)

TCS global: 23.3%


[('timestamp', 1.0),
 ('vehicle_id', 1.0),
 ('position.lat', 1.0),
 ('position.lon', 1.0),
 ('position.altitude_m', 1.0),
 ('position.heading_deg', 1.0),
 ('motion.speed_kph', 1.0),
 ('motion.bearing_deg', 0.0)]

In [5]:
# 4) Exporter en Parquet puis recharger
from telemachus.validate import to_parquet, from_parquet

out_parquet = "examples/geotab.parquet"
to_parquet(example_path, out_parquet)
df2 = from_parquet(out_parquet)
df2.head()

,timestamp,vehicle_id,events,position.lat,position.lon,position.altitude_m,position.heading_deg,motion.speed_kph,source.provider,source.device_id,source.ingest_timestamp
0,2025-01-01T12:00:00Z,GEOTAB-01,"[{'end': '2025-01-01T12:00:12Z', 'severity': '...",48.85,2.35,35.0,180.0,45.2,geotab,DEVICE-9876,2025-01-01T12:00:05Z
